In [ ]:
import os
from groq import Groq
from google import genai

# Initialize clients using environment variables
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

def generate_response(query: str, context: str) -> str:
    """A standard Groq LLM call using the provided context."""
    prompt = f"Context:\n{context}\n\nQuery:\n{query}\n\nAnswer the query using only the context provided above."
    
    chat_completion = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama3-70b-8192" 
    )
    return chat_completion.choices[0].message.content

In [ ]:
def get_gemini_embedding(text: str) -> list[float]:
    """Generates an embedding vector using Gemini."""
    result = gemini_client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    return result.embeddings[0].values

def basic_rag(query: str, vector_db) -> str:
    # 1. Embed the user's query
    query_embedding = get_gemini_embedding(query)
    
    # 2. Retrieve top matching documents (Mocking the vector search logic)
    # E.g., Pinecone, Astra DB, or Milvus
    retrieved_chunks = vector_db.similarity_search(query_embedding, k=3) 
    context = "\n".join(retrieved_chunks)
    
    # 3. Generate the answer via Groq
    return generate_response(query, context)